# Stage 6: a validated pricing agent — the model chooses, the code calculates and checks

The earlier stages each left one problem open:
- **Stage 3** grounds its price in a knowledge base, but the arithmetic and the pricing rules are left to the model. One answer priced a luxury handbag *below* the competitor while citing the guideline that says to price at or above it, and another reported a 30.8% margin that was really 30.7%.
- **Stage 4** made the arithmetic exact with calculation tools, but those tools were never connected to the knowledge base, and their rules (a fixed 5% undercut, a 10% premium) were not the knowledge base's own.
- **Stage 5** checks its answers in a LangGraph loop, but every check is an LLM's judgement, and the checks disagreed with themselves on identical text.

This stage puts them together, with one principle: **the model chooses, the code calculates and checks.**

```
product details (fields, or a message parsed into fields and checked by code)
   │
retrieve ── stage 3's filtered searches, plus the two guidelines the code enforces
   │
grade evidence ── a model checks only what similarity picked (past decisions, general guidelines)
   │
calculate (code) ── margin floor · the band the elasticity guideline allows · the allowed price points
   │
decide (model) ── structured output: one price from the allowed list, with cited reasons
   │
check rules (code) ──── fail ──► decide again, told why (3 attempts; then the rule-based price)
   │ pass
check the explanation against the evidence (model) ── fail ──► decide again (3 attempts; then LOW CONFIDENCE)
   │ pass
answer ── every number written by code ──► live scores: code checks + an LLM judge, logged as JSONL
```

All the code lives in [`validated_pricing_agent.py`](validated_pricing_agent.py), which also runs from the command line. This notebook walks through it and measures it.

**Before you run it:** it needs `CEREBRAS_API_KEY` or `GROQ_API_KEY` in the `.env` file at the repository root. A full run makes about 130 model calls (roughly 170,000 tokens): under 2 minutes on Cerebras. On Groq's free tier, set `N_RUNS = 1` in section 3 to stay well within the daily token limit.

In [1]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")  # keep the model-loading bars out of the output
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
warnings.filterwarnings("ignore", message="IProgress not found")

import pandas as pd
from langchain_core.runnables import RunnableLambda

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

sys.path.insert(0, str(Path.cwd()))  # the notebook runs from this folder
import validated_pricing_agent as vpa

name, model = vpa.provider()
print(f"Model: {model} on {name}")

embeddings = vpa.kb.make_embeddings()                             # local all-MiniLM-L6-v2, shared
knowledge = vpa.KnowledgeBase(embeddings)                          # stage 3's knowledge base
knowledge_plus = vpa.KnowledgeBase(embeddings, include_new_category=True,
                                   name="validated_pricing_plus")  # the same, plus the Sports & Fitness data
print(f"Knowledge base: {len(knowledge.store.get()['ids'])} documents "
      f"({len(knowledge_plus.store.get()['ids'])} with the Sports & Fitness data)")

Model: gpt-oss-120b on Cerebras
Knowledge base: 52 documents (60 with the Sports & Fitness data)


## 1. The calculations, before any model call

For every product the code works out, from the inputs and the retrieved evidence:
- **the margin floor**: `cost / (1 − target margin)`, stage 4's `margin_calculator`. Nothing may be priced below it.
- **the band around the competitor** that the knowledge base's guideline for the product's elasticity allows: within 3% for High, within 5% for Medium, at or up to 10% above for Low, and within 5% for a category with no history. The elasticity comes from the merchant or, if not given, from the category's benchmark.
- **the allowed price points**: mass-market prices end in 49 or 99 and luxury prices are round, as the knowledge base's price-point guideline says.
- **the rule-based default**: the lowest allowed price, used if the model keeps breaking the rules.

These are the seven test products. No model is called in this section.

In [2]:
PRODUCTS = {
    "Puma sneakers": dict(product_name="Puma Sneakers", category="Footwear", cost_price=1800, target_margin=30,
                          competitor_price=2699),
    "Headphones": dict(product_name="Wireless Noise-Cancelling Headphones", category="Electronics", cost_price=22000,
                       target_margin=18, competitor_price=26990, current_price=27990),
    "Handbag": dict(product_name="Coach Leather Handbag", category="Luxury Goods", cost_price=16000, target_margin=55,
                    competitor_price=38500),
    "Speaker": dict(product_name="JBL Bluetooth Speaker", category="Electronics", cost_price=2400, target_margin=25,
                    competitor_price=2999),
    "Pressure cooker": dict(product_name="Prestige Pressure Cooker 5L", category="Home & Kitchen", cost_price=1500,
                            target_margin=38),
    "Exercise bike": dict(product_name="Magnetic Exercise Bike", category="Sports & Fitness", cost_price=9800,
                          target_margin=30, competitor_price=14499),
    "Exercise bike + data": dict(product_name="Magnetic Exercise Bike", category="Sports & Fitness", cost_price=9800,
                                 target_margin=30, competitor_price=14499),
}
KNOWLEDGE_FOR = {key: (knowledge_plus if key == "Exercise bike + data" else knowledge) for key in PRODUCTS}
WHAT_IT_TESTS = {
    "Puma sneakers": "the reference product: High elasticity, room above the floor",
    "Headphones": "a floor just under the competitor's price; a current price to compare with",
    "Handbag": "Low elasticity and round luxury prices (stage 3 priced it below the competitor)",
    "Speaker": "a competitor priced below the margin floor: the rules conflict",
    "Pressure cooker": "no competitor price, so no band",
    "Exercise bike": "a category with no data in the knowledge base",
    "Exercise bike + data": "the same bike after the Sports & Fitness data is added",
}

for key, product in PRODUCTS.items():
    request = vpa.PricingRequest(**product)
    options = vpa.compute_options(request, KNOWLEDGE_FOR[key].retrieve(request))
    print(f"{key}: {WHAT_IT_TESTS[key]}")
    print(options.describe(request))
    print()

Puma sneakers: the reference product: High elasticity, room above the floor
- Margin floor: ₹1,800 / (1 - 30%) = ₹2,571.43; nothing below it.
- Elasticity: High (from the category benchmark).
- Competitor price ₹2,699; allowed band ₹2,618.03 to ₹2,779.97: within 3% of the competitor (High elasticity guideline).
- Price points: prices end in 49 or 99.
- Allowed prices: ₹2,649, ₹2,699, ₹2,749.

Headphones: a floor just under the competitor's price; a current price to compare with
- Margin floor: ₹22,000 / (1 - 18%) = ₹26,829.27; nothing below it.
- Elasticity: High (from the category benchmark).
- Competitor price ₹26,990; allowed band ₹26,180.30 to ₹27,799.70: within 3% of the competitor (High elasticity guideline).
- Price points: prices end in 49 or 99.
- Allowed prices: ₹26,849, ₹26,899, ₹26,949, ₹26,999, ₹27,049, ₹27,099, ..., ₹27,699, ₹27,749, ₹27,799 (20 in all).

Handbag: Low elasticity and round luxury prices (stage 3 priced it below the competitor)
- Margin floor: ₹16,000 / (1 

**What this shows**
- **The sneakers have only three allowed prices** (₹2,649, ₹2,699, ₹2,749): the margin floor (₹2,571.43) cuts off the bottom of the ±3% band. The model's whole job is to pick one and say why.
- **The speaker's rules conflict.** The floor (₹3,200) is above the entire High-elasticity band (₹2,909-₹3,089), so the code allows exactly one price, ₹3,249, and the answer must state the trade-off.
- **The pressure cooker has no competitor price,** so only the floor and the price-point rule apply.
- **The exercise bike's category has no data,** so the no-history guideline's ±5% band applies. With the Sports & Fitness data added, the band comes from that category's benchmark, which says Medium: also ±5% (section 7 returns to this).
- The headphones' competitor price, ₹26,990, is not itself an allowed price: it ends in 90, not 49 or 99.

## 2. One product, step by step

The reference product, with the trace of every step. The decision comes back as a `PricingDecision` object (structured output), not as free text; the answer is then written by code from it.

In [3]:
agent = vpa.ValidatedPricingAgent(knowledge)
first = agent.price(PRODUCTS["Puma sneakers"])
print("\n" + first.answer)

---RETRIEVE (stage 3's filtered searches)---
---GRADE EVIDENCE (5 picked by similarity; 6 picked by metadata, kept)---
   keep: decision:Puma Smash Sneakers
   keep: decision:Puma Softride Running Shoes
   keep: decision:Store-brand Canvas Sneakers
   keep: decision:Adidas Ultraboost Running Shoes
   drop: guideline:Festive and sale periods
---CALCULATE (code)---
- Margin floor: ₹1,800 / (1 - 30%) = ₹2,571.43; nothing below it.
- Elasticity: High (from the category benchmark).
- Competitor price ₹2,699; allowed band ₹2,618.03 to ₹2,779.97: within 3% of the competitor (High elasticity guideline).
- Price points: prices end in 49 or 99.
- Allowed prices: ₹2,649, ₹2,699, ₹2,749.
---DECIDE (model, structured output) attempt 1 of 3---
   chose ₹2,649
---CHECK RULES (code)---
   all rules pass
---CHECK THE EXPLANATION AGAINST THE EVIDENCE (model)---
   NOT supported: "₹2,649 is 2.5% below the main competitor’s price"
---DECIDE (model, structured output) attempt 2 of 3---
   chose ₹2,649
---C

In [4]:
print("The model's structured decision:\n")
print(first.state["decision"].model_dump_json(indent=2))
print("\nCode checks on the final answer:", first.code_checks)
print("Judge:", {k: v for k, v in first.judge.items() if not k.endswith("_reason")})
print(f"Path: {' -> '.join(first.path)}")
print(f"{first.calls} model calls (+{first.judge_calls} for the judge), {first.tokens + first.judge_tokens:,} tokens, "
      f"{first.latency_s:.1f} s")

The model's structured decision:

{
  "recommended_price": 2649.0,
  "evidence": [
    {
      "doc": 8,
      "point": "Guideline for high‑elasticity categories: stay within ±3% of the main competitor and prefer a slight discount when margin permits"
    },
    {
      "doc": 4,
      "point": "Past Puma pricing set ~4% below the competitor achieved >30% margin and sold 92% of stock at full price"
    },
    {
      "doc": 9,
      "point": "Psychological pricing: ends in 49 or 99 to signal value"
    },
    {
      "doc": 3,
      "point": "Competitor trend shows entry‑level branded sneakers cluster between ₹2,199‑₹2,999"
    }
  ],
  "rationale": "₹2,649 ends in 49, meets the psychological price‑point guideline, and sits comfortably within the ±3% band around the competitor’s ₹2,699 price, mirroring a prior successful Puma pricing that was set slightly below the competitor while preserving the target margin. The price also falls within the typical market cluster for entry‑level bran

**What this shows**
- **The first decision was sent back.** Its explanation said ₹2,649 is "2.5% below" the competitor; it is 1.9%. The model was told not to calculate, calculated anyway and got it wrong, and the grounding check caught the number. The second attempt kept the same price and left the percentage out.
- **The answer's numbers come from code:** the 32.0% margin and the 1.9% gap are calculated from the chosen price, so they can't be wrong in the way the rejected explanation was.
- **The structured decision is what the code checks:** a price that must be on the allowed list, and evidence as document numbers that must exist.
- One product took 9 model calls (5 relevance grades, 2 decisions, 2 grounding checks) plus 1 for the judge, about 4 seconds.

## 3. The test set

Every product runs `N_RUNS` times, and every answer is scored as it is produced: the code checks are re-run on the final answer, and the LLM judge rates the decision (section 7 looks at the scores). The scores are also appended to a JSONL log, as they would be in production.

In [5]:
N_RUNS = 2
LOG_PATH = Path(tempfile.gettempdir()) / "validated_pricing_log.jsonl"
LOG_PATH.unlink(missing_ok=True)

agents = {id(k): vpa.ValidatedPricingAgent(k, log_path=LOG_PATH, verbose=False) for k in (knowledge, knowledge_plus)}
results = []
for key, product in PRODUCTS.items():
    for run in range(1, N_RUNS + 1):
        result = agents[id(KNOWLEDGE_FOR[key])].price(product, run_label=f"{key} #{run}")
        results.append((key, run, result))
        print(f"{key:21s} run {run}: {vpa.kb.money(result.final_price):>8s}  {result.status:14s} "
              f"attempts {result.attempts}, {result.calls} calls, {result.latency_s:.1f} s")

table = pd.DataFrame([{"case": key, "run": run, **r.summary()} for key, run, r in results]).drop(columns="product")
table

Puma sneakers         run 1:   ₹2,649  ok             attempts 2, 9 calls, 4.5 s
Puma sneakers         run 2:   ₹2,649  ok             attempts 2, 9 calls, 3.9 s
Headphones            run 1:  ₹26,899  ok             attempts 1, 6 calls, 2.3 s
Headphones            run 2:  ₹26,899  ok             attempts 1, 6 calls, 2.4 s
Handbag               run 1:  ₹39,000  ok             attempts 1, 6 calls, 1.8 s
Handbag               run 2:  ₹39,000  ok             attempts 1, 6 calls, 2.1 s
Speaker               run 1:   ₹3,249  ok             attempts 1, 7 calls, 2.5 s
Speaker               run 2:   ₹3,249  ok             attempts 1, 7 calls, 2.4 s
Pressure cooker       run 1:   ₹2,499  ok             attempts 1, 6 calls, 2.3 s
Pressure cooker       run 2:   ₹2,449  ok             attempts 1, 6 calls, 2.1 s
Exercise bike         run 1:  ₹14,049  ok             attempts 1, 3 calls, 1.9 s
Exercise bike         run 2:  ₹14,049  ok             attempts 1, 3 calls, 2.1 s
Exercise bike + data  run 1:

,case,run,price,margin_%,vs_competitor_%,status,attempts,dropped,rejected_by,code_checks,judge,calls,tokens,seconds
0,Puma sneakers,1,2649,32.0,-1.9,ok,2,1,grounding x1,5/5,3/3,9,12186,4.5
1,Puma sneakers,2,2649,32.0,-1.9,ok,2,1,grounding x1,5/5,3/3,9,12128,3.9
2,Headphones,1,26899,18.2,-0.3,ok,1,3,-,5/5,3/3,6,6555,2.3
3,Headphones,2,26899,18.2,-0.3,ok,1,3,-,5/5,3/3,6,6559,2.4
4,Handbag,1,39000,59.0,1.3,ok,1,3,-,5/5,3/3,6,5771,1.8
5,Handbag,2,39000,59.0,1.3,ok,1,3,-,5/5,3/3,6,5772,2.1
6,Speaker,1,3249,26.1,8.3,ok,1,5,-,5/5,3/3,7,6857,2.5
7,Speaker,2,3249,26.1,8.3,ok,1,5,-,5/5,3/3,7,6857,2.4
8,Pressure cooker,1,2499,40.0,NaN,ok,1,1,-,5/5,3/3,6,6497,2.3
9,Pressure cooker,2,2449,38.8,NaN,ok,1,1,-,5/5,3/3,6,6252,2.1


In [6]:
print("Why decisions were sent back:\n")
for key, run, result in results:
    for reason in result.rejections:
        print(f"- {key} run {run}: {reason}")

Why decisions were sent back:

- Puma sneakers run 1: grounding: "₹2,649 is 2.5% below the main competitor’s price"
- Puma sneakers run 2: grounding: "₹2,649 is 2.5% below the main competitor’s price"


**What this shows**
- **Every answer kept every rule** (5/5 code checks on all 14), and 12 of 14 passed on the first attempt.
- **The two retries were the sneakers again,** for the same miscalculated "2.5%". The rules never had to send a decision back in this run: choosing from a list of allowed prices leaves little room to break them.
- **The same product mostly gets the same price:** 5 of the 7 cases priced identically on both runs. The two that differed (the pressure cooker, ₹2,499 or ₹2,449, and the bike with data, ₹14,499 or ₹14,049) have no competitor price or a wide band, so more prices are allowed.
- **The cost:** 3 to 9 model calls per answer, plus 1 for the judge; about 2-4.5 seconds each.

In [7]:
for key, run, result in results:
    if run == 1:
        print("=" * 100)
        print(f"{key}  ({WHAT_IT_TESTS[key]})")
        print("-" * 100)
        print(result.answer)

Puma sneakers  (the reference product: High elasticity, room above the floor)
----------------------------------------------------------------------------------------------------
Recommended price: ₹2,649
- Margin: 32.0% (target 30%, floor ₹2,571.43)
- Competitor: ₹2,699; this price is 1.9% below it
- Allowed: ₹2,618.03 to ₹2,779.97, within 3% of the competitor (High elasticity guideline)
Why: ₹2,649 ends in 49, meets the psychological price‑point guideline, and sits comfortably within the ±3% band around the competitor’s ₹2,699 price, mirroring a prior successful Puma pricing that was set slightly below the competitor while preserving the target margin. The price also falls within the typical market cluster for entry‑level branded sneakers, supporting competitive positioning.
Risks: Watch for any rapid competitor price cuts that could erode the advantage of the modest discount.
Sources: [3] trend:Footwear, [4] decision:Puma Smash Sneakers, [8] guideline:High elasticity: small price ch

**What this shows**
- **The explanations use the evidence the way stage 3's did:** the headphones avoid the Sony headphones' mistake (priced above a competitor, sales 20% below plan); the handbag follows the Michael Kors decision and stays above the competitor, as the Low elasticity guideline asks.
- **The speaker's answer handles the conflict:** it keeps the floor, says the price sits above the competitor band, and states the trade-off (compete on value). Stage 3's answer for the same product was ₹3,200, the floor itself, which is not a price point (section 6).
- **The bike with no category data is priced at the first price point above the floor,** as the no-history guideline says ("start at the floor price").

## 4. When the model breaks a rule, or its explanation is not supported

The loop is only as good as what happens when a check keeps failing. Two tests with stand-ins for the model, so they make **no API calls**:
- a decider that always answers ₹36,000 for the handbag (stage 3's price, below the competitor);
- a grounding check that never accepts the explanation.

In [8]:
offline = vpa.ValidatedPricingAgent(knowledge, judge=False)
offline.grader = RunnableLambda(lambda inputs: vpa.GradeDocument(binary_score="yes"))  # keep every document


def stage3_handbag_decision(inputs):
    return vpa.PricingDecision(
        recommended_price=36000, evidence=[vpa.EvidenceUse(doc=1, point="Luxury Goods: low elasticity")],
        rationale="A price slightly below the competitor wins volume [1].", trade_off="",
        risks="Watch full-price sell-through.")


offline.decider = RunnableLambda(stage3_handbag_decision)
calls_before = vpa.USAGE.calls
broken = offline.price(PRODUCTS["Handbag"])
print("\n" + broken.answer)
print(f"\nModel calls: {vpa.USAGE.calls - calls_before}")

---RETRIEVE (stage 3's filtered searches)---
---GRADE EVIDENCE (4 picked by similarity; 6 picked by metadata, kept)---
   keep: decision:Michael Kors Leather Handbag
   keep: decision:Hugo Boss Perfume 100ml
   keep: decision:Tissot Automatic Watch
   keep: decision:Ray-Ban Sunglasses
---CALCULATE (code)---
- Margin floor: ₹16,000 / (1 - 55%) = ₹35,555.56; nothing below it.
- Elasticity: Low (from the category benchmark).
- Competitor price ₹38,500; allowed band ₹38,500 to ₹42,350: at or up to 10% above the competitor (Low elasticity guideline).
- Price points: round prices (luxury goods).
- Allowed prices: ₹38,500, ₹39,000, ₹39,500, ₹40,000, ₹40,500, ₹41,000, ₹41,500, ₹42,000.
---DECIDE (model, structured output) attempt 1 of 3---
   chose ₹36,000
---CHECK RULES (code)---
   FAIL: ₹36,000 is outside the allowed band ₹38,500 to ₹42,350 (at or up to 10% above the competitor (Low elasticity guideline))
---DECIDE (model, structured output) attempt 2 of 3---
   chose ₹36,000
---CHECK RULES

In [9]:
def puma_decision(inputs):
    return vpa.PricingDecision(
        recommended_price=2649, evidence=[vpa.EvidenceUse(doc=1, point="Footwear: High elasticity")],
        rationale="Footwear shoppers are price-sensitive [1].", trade_off="", risks="Watch weekly sales.")


offline.decider = RunnableLambda(puma_decision)
offline.grounding = RunnableLambda(lambda inputs: vpa.GroundingVerdict(
    binary_score="no", unsupported="(a stand-in check that never accepts)"))
calls_before = vpa.USAGE.calls
unsupported = offline.price(PRODUCTS["Puma sneakers"])
print("\n" + unsupported.answer)
print(f"\nModel calls: {vpa.USAGE.calls - calls_before}")

---RETRIEVE (stage 3's filtered searches)---
---GRADE EVIDENCE (5 picked by similarity; 6 picked by metadata, kept)---
   keep: decision:Puma Smash Sneakers
   keep: decision:Puma Softride Running Shoes
   keep: decision:Store-brand Canvas Sneakers
   keep: decision:Adidas Ultraboost Running Shoes
   keep: guideline:Festive and sale periods
---CALCULATE (code)---
- Margin floor: ₹1,800 / (1 - 30%) = ₹2,571.43; nothing below it.
- Elasticity: High (from the category benchmark).
- Competitor price ₹2,699; allowed band ₹2,618.03 to ₹2,779.97: within 3% of the competitor (High elasticity guideline).
- Price points: prices end in 49 or 99.
- Allowed prices: ₹2,649, ₹2,699, ₹2,749.
---DECIDE (model, structured output) attempt 1 of 3---
   chose ₹2,649
---CHECK RULES (code)---
   all rules pass
---CHECK THE EXPLANATION AGAINST THE EVIDENCE (model)---
   NOT supported: (a stand-in check that never accepts)
---DECIDE (model, structured output) attempt 2 of 3---
   chose ₹2,649
---CHECK RULES (c

**What this shows**
- **A decider that keeps breaking a rule cannot get its price through.** After 3 rejections the agent answered with the lowest price the rules allow (₹38,500, at the competitor) and labelled it RULE-BASED PRICE. Stage 3's below-competitor handbag price is exactly what this catches.
- **An explanation that is never accepted keeps its price but is flagged** LOW CONFIDENCE: the price passed every rule, so it is safe; only the reasoning is in doubt.
- Neither test called a model: the decider, the grader and the grounding check were replaced with stand-ins.

## 5. Plain-language requests

Merchants don't fill in forms. `parse_request()` asks the model to copy the details from a message into a schema, then **code checks that every number it returns really appears in the message**: a guessed number is dropped and treated as missing. If something required is missing, the agent asks for it instead of pricing.

In [10]:
MESSAGES = [
    "I sell Puma running sneakers. Myntra has them at Rs 2,699 and I want a 30% margin. What should I charge?",
    "New stock: Puma sneakers, they cost me ₹1,800 each. Competitor sells at ₹2,699, I need 30% margin. Price?",
]
for message in MESSAGES:
    print("Message:", message)
    request, problems = vpa.parse_request(message)
    if request is None:
        print("  Not priced. " + " ".join(problems))
    else:
        print("  Parsed:", request.model_dump(exclude_none=True))
        for p in problems:
            print("  note:", p)
    print()

Message: I sell Puma running sneakers. Myntra has them at Rs 2,699 and I want a 30% margin. What should I charge?
  Not priced. Please give the cost price.

Message: New stock: Puma sneakers, they cost me ₹1,800 each. Competitor sells at ₹2,699, I need 30% margin. Price?
  Parsed: {'product_name': 'Puma sneakers', 'category': 'Footwear', 'cost_price': 1800.0, 'target_margin': 30.0, 'competitor_price': 2699.0}



**What this shows**
- **The first message has no cost price,** so the agent asks for it instead of pricing. A price can't be checked against a floor it doesn't know.
- **The second message parses into a valid request,** with the category mapped to the knowledge base's "Footwear".
- Every number in a parsed request is checked against the message in code, so a value the model guessed would be dropped and reported, not used.

## 6. Compared with stage 3, under the same checks

Stage 3's agent (its own prompt and settings, the same model and provider) priced the same products. Its answers are free text, so a structured-output call pulls out the recommended price and the margin it states, and the same code checks are applied. One run per product.

In [11]:
from typing import Optional

from pydantic import BaseModel, Field


class StatedPrice(BaseModel):
    recommended_price: Optional[float] = Field(description="The single final recommended selling price in rupees, or null")
    stated_margin: Optional[float] = Field(description="The gross margin in percent the answer states for that price, or null")


stage3_llm = vpa.make_llm(max_tokens=2048, reasoning_effort="medium", temperature=0.2)  # stage 3's settings
extractor = vpa.make_llm(max_tokens=1024).with_structured_output(StatedPrice)
comparison = []
for key, product in PRODUCTS.items():
    request = vpa.PricingRequest(**product)
    stage3 = vpa.kb.RAGPricingAgent(KNOWLEDGE_FOR[key].store, llm=stage3_llm)
    answer3 = stage3.recommend(**product)
    stated = extractor.invoke(f"Extract the final recommended price and the margin stated for it.\n\n{answer3}")
    options = vpa.compute_options(request, KNOWLEDGE_FOR[key].retrieve(request))
    problems = vpa.check_price(request, options, stated.recommended_price)
    if stated.recommended_price and stated.stated_margin is not None:
        actual = vpa.margin_percent(stated.recommended_price, request.cost_price)
        decimals = len(f"{stated.stated_margin:g}".partition(".")[2])
        if round(actual, decimals) != round(stated.stated_margin, decimals):
            problems.append(f"it states a {stated.stated_margin:g}% margin, but the price gives {actual:.2f}%")
    stage6 = next(r for k, run, r in results if k == key and run == 1)
    comparison.append({"case": key,
                       "stage 3": vpa.kb.money(stated.recommended_price) if stated.recommended_price else "none",
                       "stage 3 rules broken": len(problems),
                       "stage 6": vpa.kb.money(stage6.final_price),
                       "stage 6 rules broken": len(vpa.check_price(request, stage6.state["options"], stage6.final_price)),
                       "problems": problems})

print(pd.DataFrame(comparison).drop(columns="problems").to_string(index=False))
print("\nWhat the stage 3 answers got wrong:")
for row in comparison:
    for problem in row["problems"]:
        print(f"- {row['case']}: {problem}")

                case stage 3  stage 3 rules broken stage 6  stage 6 rules broken
       Puma sneakers  ₹2,599                     2  ₹2,649                     0
          Headphones ₹26,899                     0 ₹26,899                     0
             Handbag ₹40,000                     0 ₹39,000                     0
             Speaker  ₹3,200                     2  ₹3,249                     0
     Pressure cooker  ₹2,420                     1  ₹2,499                     0
       Exercise bike ₹14,899                     1 ₹14,049                     0
Exercise bike + data ₹14,099                     0 ₹14,499                     0

What the stage 3 answers got wrong:
- Puma sneakers: ₹2,599 is outside the allowed band ₹2,618.03 to ₹2,779.97 (within 3% of the competitor (High elasticity guideline))
- Puma sneakers: it states a 30.8% margin, but the price gives 30.74%
- Speaker: the floor and the competitor band conflict, so the price must be ₹3,249
- Speaker: ₹3,200 is not a pr

**What this shows**
- **Stage 3 broke at least one rule on 4 of the 7 products;** stage 6 broke none.
  - **Sneakers:** ₹2,599 is 3.7% below the competitor, outside the High elasticity guideline's 3%, and the answer states a 30.8% margin for what is 30.74%.
  - **Speaker:** ₹3,200 is the floor itself, which is not a price point; the first price point above the floor is ₹3,249.
  - **Pressure cooker:** ₹2,420 is not a price point.
  - **Exercise bike:** the answer states a 32.4% margin for ₹14,899, which gives 34.22%.
- **Stage 3's answers were not bad advice.** Most are close to stage 6's prices. But each rule held only when the model remembered it, and the arithmetic was right only when the model got it right. In stage 6, code does both.
- Stage 3's handbag was fine this time (₹40,000); in the run saved in stage 3's own notebook it was ₹36,000, below the competitor.

## 7. Live evaluation

Every answer above was scored as it was produced and logged. In production the same log would be read on a schedule, and a falling pass rate, more retries or more fallbacks would show that something changed: the model, the prompt, the knowledge base or the traffic.

- **Code checks** (every answer): above the floor, inside the band, a valid price point, the margin in the answer correct, citations that exist.
- **LLM judge** (every answer here; a sample would do in production): does the price follow the evidence, is the explanation complete, is it clear. It is a monitor, not a gate: the answer has already been returned.

In [12]:
log = pd.read_json(LOG_PATH, lines=True)
checks = pd.json_normalize(log["code_checks"])
judge = pd.json_normalize(log["judge"])
print(f"{len(log)} answers logged to {LOG_PATH.name}\n")
print("Code checks passed:")
print((checks.mean() * 100).round(0).astype(int).astype(str).add("%").to_string())
print("\nJudge criteria passed:")
print((judge[["follows_evidence", "complete", "clear"]].mean() * 100).round(0).astype(int).astype(str).add("%").to_string())
print("\nStatus:", log["status"].value_counts().to_dict())
print("Attempts:", log["attempts"].value_counts().sort_index().to_dict())
print(f"Per answer: {log['calls'].mean():.1f} model calls, {log['tokens'].mean():,.0f} tokens, "
      f"{log['latency_s'].mean():.1f} s (plus the judge)")

14 answers logged to validated_pricing_log.jsonl

Code checks passed:
above_floor                 100%
in_band                     100%
price_point                 100%
margin_in_answer_correct    100%
citations_valid             100%

Judge criteria passed:
follows_evidence     86%
complete            100%
clear               100%

Status: {'ok': 14}
Attempts: {1: 12, 2: 2}
Per answer: 6.3 model calls, 6,998 tokens, 2.6 s (plus the judge)


In [13]:
failed = [(row["label"], c.replace("_reason", ""), row["judge"][c]) for _, row in log.iterrows()
          for c in ("follows_evidence_reason", "complete_reason", "clear_reason")
          if not row["judge"][c.replace("_reason", "")]]
print(f"Judge fails: {len(failed)}")
for label, criterion, reason in failed:
    print(f"- {label}, {criterion}: {reason}")

Judge fails: 2
- Exercise bike + data #1, follows_evidence: The recommendation treats the bike as medium elasticity despite evidence that big‑ticket equipment like exercise bikes shows high elasticity, so it does not align with the provided evidence.
- Exercise bike + data #2, follows_evidence: The evidence indicates big‑ticket equipment like exercise bikes has high elasticity, yet the recommendation applies a medium‑elasticity guideline, contradicting the benchmark.


**What this shows**
- **The code checks are a wall, not a measure:** 100% because a failing price never leaves the agent. In production their job is to stay at 100%; a drop would mean the code itself changed.
- **The judge found the one real weakness, on both runs:** with the Sports & Fitness data, the band comes from the category benchmark's label (Medium, ±5%), but the same benchmark's notes say big-ticket equipment such as exercise bikes behaves like High elasticity (±3%). The second run's ₹14,049 is 3.1% below the competitor: allowed by the code, outside the High elasticity guideline. The code took the category label; the judge read the notes.
- **The fix belongs in the data or the request, not the prompt:** benchmarks per sub-category, or the merchant passing `price_elasticity="High"` for big-ticket items.
- **The log is what makes this a live evaluation:** the same pass rates, retries and fallbacks, read every day, would show a model update, a prompt edit or new knowledge changing the agent's behaviour.

## Summary

- **The model chooses, the code calculates and checks.** The margin floor, the guideline's band around the competitor and the price points are all code; the model picks one allowed price and explains it as structured output.
- **No answer broke a pricing rule** (14 of 14), which holds by construction: a failing price never leaves the agent. Stage 3, given the same products, broke at least one rule on 4 of 7: prices outside the guideline's band, prices that are not price points, and margins misstated by up to 1.8 points.
- **The checks catch what each is good at.** Code enforces the rules. The grounding check caught the model's own arithmetic (a "2.5%" that was 1.9%). The judge caught a data problem: a category-level elasticity label that hides a sub-category's different behaviour.
- **It fails closed.** A price that keeps breaking a rule is replaced by the rule-based price and labelled; an explanation that is never accepted is flagged LOW CONFIDENCE.
- **The cost:** about 6 model calls (7,000 tokens) and 2.6 seconds per answer, plus the judge. Most of the calls are relevance grades, which run in parallel.